In [3]:
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

# FeedForward with SwiGLU activation
class FeedForward(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        multiple_of: int,
        ffn_dim_multiplier: Optional[float],
    ):
        super().__init__()
        hidden_dim = int(2 * hidden_dim / 3)
        # custom dim factor multiplier
        if ffn_dim_multiplier is not None:
            hidden_dim = int(ffn_dim_multiplier * hidden_dim)
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden_dim, bias=False) # GateLinear
        self.w2 = nn.Linear(hidden_dim, dim, bias=False) # DownLinear
        self.w3 = nn.Linear(dim, hidden_dim, bias=False) # UpLinear

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))
    

# 测试示例
if __name__ == "__main__":
    layer = FeedForward(dim=128, hidden_dim=256, multiple_of=8, ffn_dim_multiplier=None)
    x = torch.randn(1, 128)
    out = layer(x)
    print(out.shape)  # torch.Size([1, 128])

torch.Size([1, 128])


In [2]:
import torch
import torch.nn as nn

class LlamaRMSNorm(nn.Module):
    """LLaMA-style RMSNorm"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        # hidden_states: [batch_size, seq_len, dim]
        input_dtype = hidden_states.dtype
        hidden_states_fp32 = hidden_states.to(torch.float32)
        variance = torch.mean(hidden_states_fp32 ** 2, dim=-1, keepdim=True)
        normed = hidden_states_fp32 * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * normed.to(input_dtype)

# 测试 RMSNorm
batch_size, seq_len, dim = 2, 4, 8
x = torch.randn(batch_size, seq_len, dim)

# 你的实现
rmsnorm_custom = LlamaRMSNorm(dim, eps=1e-6)
output_custom = rmsnorm_custom(x)

# PyTorch 实现（同 eps）
rmsnorm_pytorch = nn.RMSNorm(dim, eps=1e-6)
# 为了公平比较，把权重对齐
with torch.no_grad():
    rmsnorm_pytorch.weight.copy_(rmsnorm_custom.weight)

output_pytorch = rmsnorm_pytorch(x)

print("输入和输出的形状:", x.shape, output_custom.shape)
if torch.allclose(output_custom, output_pytorch, atol=1e-6):
    print("结果验证通过: 自定义 RMSNorm 与 PyTorch nn.RMSNorm 一致！")
else:
    print("结果验证失败: 两者结果不一致。")
    print("max abs diff =", (output_custom - output_pytorch).abs().max().item())

输入和输出的形状: torch.Size([2, 4, 8]) torch.Size([2, 4, 8])
结果验证通过: 自定义 RMSNorm 与 PyTorch nn.RMSNorm 一致！
